In [16]:
import os
import sys
from decimal import Decimal
from pyspark.sql.functions import sum, count, broadcast

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    DecimalType,
    StringType,
    StructField,
    StructType,
)

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("FinancialDataLab")
    .config("spark.driver.host", "localhost")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

In [2]:
transaction_schema = StructType([
    StructField("TransactionId", StringType(), False),
    StructField("AccountId", StringType(), False),
    StructField("Amount", DecimalType(18, 2), True),
])

transactions = [
    ("T001", "A100", Decimal("250.00")),
    ("T002", "A101", Decimal("100.00")),
    ("T003", "A102", Decimal("-50.00")),
    ("T004", "A103", Decimal("500.00")),
    ("T005", "A104", Decimal("75.00")),
]

In [3]:
transactions_df = spark.createDataFrame(transactions, schema=transaction_schema)
transactions_df.printSchema()
transactions_df.show()

root
 |-- TransactionId: string (nullable = false)
 |-- AccountId: string (nullable = false)
 |-- Amount: decimal(18,2) (nullable = true)

+-------------+---------+------+
|TransactionId|AccountId|Amount|
+-------------+---------+------+
|         T001|     A100|250.00|
|         T002|     A101|100.00|
|         T003|     A102|-50.00|
|         T004|     A103|500.00|
|         T005|     A104| 75.00|
+-------------+---------+------+



In [4]:
negative_transactions_df = transactions_df.filter(transactions_df.Amount < 0)
negative_transactions_df.show()

+-------------+---------+------+
|TransactionId|AccountId|Amount|
+-------------+---------+------+
|         T003|     A102|-50.00|
+-------------+---------+------+



In [5]:
positive_transactions_df = transactions_df.filter(transactions_df.Amount >= 0)

In [6]:
positive_transactions_df.show()

+-------------+---------+------+
|TransactionId|AccountId|Amount|
+-------------+---------+------+
|         T001|     A100|250.00|
|         T002|     A101|100.00|
|         T004|     A103|500.00|
|         T005|     A104| 75.00|
+-------------+---------+------+



In [7]:
additional_transactions = [
    ("T006", "A100", Decimal("50.00")),   
    ("T007", "A101", Decimal("25.00")),
    ]

additional_transactions_df = spark.createDataFrame(additional_transactions, schema=transaction_schema)
transactions_with_repeats_df = transactions_df.union(additional_transactions_df)
transactions_with_repeats_df.show()


+-------------+---------+------+
|TransactionId|AccountId|Amount|
+-------------+---------+------+
|         T001|     A100|250.00|
|         T002|     A101|100.00|
|         T003|     A102|-50.00|
|         T004|     A103|500.00|
|         T005|     A104| 75.00|
|         T006|     A100| 50.00|
|         T007|     A101| 25.00|
+-------------+---------+------+



In [8]:
account_totals_df = transactions_with_repeats_df.groupBy("AccountId").agg(sum("Amount").alias("TotalAmount"),count("AccountId").alias("NumTransactions"))
account_totals_df.show()

+---------+-----------+---------------+
|AccountId|TotalAmount|NumTransactions|
+---------+-----------+---------------+
|     A100|     300.00|              2|
|     A101|     125.00|              2|
|     A102|     -50.00|              1|
|     A103|     500.00|              1|
|     A104|      75.00|              1|
+---------+-----------+---------------+



In [9]:
high_value_accounts_df = account_totals_df.filter(account_totals_df.TotalAmount > 300)
high_value_accounts_df.show()
high_value_accounts_df.explain()

+---------+-----------+---------------+
|AccountId|TotalAmount|NumTransactions|
+---------+-----------+---------------+
|     A103|     500.00|              1|
+---------+-----------+---------------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Filter (isnotnull(TotalAmount#49) AND (TotalAmount#49 > 300.00))
   +- HashAggregate(keys=[AccountId#1], functions=[sum(Amount#2), count(1)])
      +- Exchange hashpartitioning(AccountId#1, 200), ENSURE_REQUIREMENTS, [plan_id=275]
         +- HashAggregate(keys=[AccountId#1], functions=[partial_sum(Amount#2), partial_count(1)])
            +- Union
               :- Project [AccountId#1, Amount#2]
               :  +- Scan ExistingRDD[TransactionId#0,AccountId#1,Amount#2]
               +- Project [AccountId#34, Amount#35]
                  +- Scan ExistingRDD[TransactionId#33,AccountId#34,Amount#35]




In [10]:
positive_account_totals_df = transactions_with_repeats_df.filter(transactions_with_repeats_df.Amount > 0).groupBy("AccountId").agg(sum("Amount").alias("TotalAmount"))
positive_account_totals_df.show()
positive_account_totals_df.explain()

+---------+-----------+
|AccountId|TotalAmount|
+---------+-----------+
|     A100|     300.00|
|     A101|     125.00|
|     A103|     500.00|
|     A104|      75.00|
+---------+-----------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[AccountId#1], functions=[sum(Amount#2)])
   +- Exchange hashpartitioning(AccountId#1, 200), ENSURE_REQUIREMENTS, [plan_id=397]
      +- HashAggregate(keys=[AccountId#1], functions=[partial_sum(Amount#2)])
         +- Union
            :- Project [AccountId#1, Amount#2]
            :  +- Filter (isnotnull(Amount#2) AND (Amount#2 > 0.00))
            :     +- Scan ExistingRDD[TransactionId#0,AccountId#1,Amount#2]
            +- Project [AccountId#34, Amount#35]
               +- Filter (isnotnull(Amount#35) AND (Amount#35 > 0.00))
                  +- Scan ExistingRDD[TransactionId#33,AccountId#34,Amount#35]




In [11]:
account_totals_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[AccountId#1], functions=[sum(Amount#2), count(1)])
   +- Exchange hashpartitioning(AccountId#1, 200), ENSURE_REQUIREMENTS, [plan_id=426]
      +- HashAggregate(keys=[AccountId#1], functions=[partial_sum(Amount#2), partial_count(1)])
         +- Union
            :- Project [AccountId#1, Amount#2]
            :  +- Scan ExistingRDD[TransactionId#0,AccountId#1,Amount#2]
            +- Project [AccountId#34, Amount#35]
               +- Scan ExistingRDD[TransactionId#33,AccountId#34,Amount#35]




In [12]:
accounts_schema = StructType([
    StructField("AccountId", StringType(), False),
    StructField("CustomerId", StringType(), False),
    StructField("AccountType", StringType(), False),
])

accounts = [
    ("A100", "C001", "Checking"),   
    ("A101", "C002", "Savings"),
    ("A102", "C001", "Savings"),
    ("A103", "C003", "Checking"),
    ("A104", "C004", "Checking"),
]

accounts_df = spark.createDataFrame(accounts, schema=accounts_schema)
accounts_df.printSchema()
accounts_df.show()

root
 |-- AccountId: string (nullable = false)
 |-- CustomerId: string (nullable = false)
 |-- AccountType: string (nullable = false)

+---------+----------+-----------+
|AccountId|CustomerId|AccountType|
+---------+----------+-----------+
|     A100|      C001|   Checking|
|     A101|      C002|    Savings|
|     A102|      C001|    Savings|
|     A103|      C003|   Checking|
|     A104|      C004|   Checking|
+---------+----------+-----------+



In [13]:
bad_transactions = [
    ("T008", "A999", Decimal("200.00") ),  # Non-existent account
]
bad_transactions_df = spark.createDataFrame(bad_transactions, schema=transaction_schema)
transactions_for_validation_df = transactions_with_repeats_df.union(bad_transactions_df)

In [14]:
invalid_transactions_df = transactions_for_validation_df.join(accounts_df, on="AccountId", how="left_anti")
invalid_transactions_df.show()

+---------+-------------+------+
|AccountId|TransactionId|Amount|
+---------+-------------+------+
|     A999|         T008|200.00|
+---------+-------------+------+



In [17]:
valid_transactions_df = transactions_for_validation_df.join(broadcast(accounts_df), on="AccountId", how="left_semi")
valid_transactions_df.show()
valid_transactions_df.explain()

+---------+-------------+------+
|AccountId|TransactionId|Amount|
+---------+-------------+------+
|     A100|         T001|250.00|
|     A101|         T002|100.00|
|     A102|         T003|-50.00|
|     A103|         T004|500.00|
|     A104|         T005| 75.00|
|     A100|         T006| 50.00|
|     A101|         T007| 25.00|
+---------+-------------+------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Union
   :- Project [AccountId#1, TransactionId#0, Amount#2]
   :  +- BroadcastHashJoin [AccountId#1], [AccountId#116], LeftSemi, BuildRight, false, false
   :     :- Scan ExistingRDD[TransactionId#0,AccountId#1,Amount#2]
   :     +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, false]),false), [plan_id=1918]
   :        +- Project [AccountId#116]
   :           +- Scan ExistingRDD[AccountId#116,CustomerId#117,AccountType#118]
   :- Project [AccountId#34, TransactionId#33, Amount#35]
   :  +- BroadcastHashJoin [AccountId#34], [AccountId#116], LeftSe